# ViT5 inference trên 100 bài Vietnews test

Settings: GPU T4 x2, Internet On, dataset `vietnews-test100`.
Model: `VietAI/vit5-base-vietnews-summarization`

**Bắt buộc:** cell 1 gỡ transformers 5 rồi cài 4.44.2. Xong thì **Restart session**, chạy từ cell import. Đừng Run All.

In [ ]:
!nvidia-smi
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" sentencepiece protobuf
import transformers, tokenizers
print("transformers", transformers.__version__, "tokenizers", tokenizers.__version__)

Sau cell trên: **Restart session**. In ra `transformers 4.44.2` mới chạy tiếp. Nếu vẫn 4.5x thì restart chưa xong.

In [ ]:
from pathlib import Path
import json
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.44"), "Restart session sau pip, rồi chạy lại cell này"

MODEL = "VietAI/vit5-base-vietnews-summarization"
files = sorted(Path("/kaggle/input").rglob("*.txt.seg"))[:100]
print("n files", len(files))
print("first", files[0] if files else None)

In [ ]:
def parse_file(path):
    parts = [p.strip() for p in Path(path).read_text(encoding="utf-8").split("\n\n") if p.strip()]
    return {"id": Path(path).name, "title": parts[0], "abstract": parts[1], "body": "\n".join(parts[2:])}

docs = [parse_file(p) for p in files]
print(docs[0]["id"], len(docs[0]["body"].split()), "n=", len(docs))

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device", device)

tok = T5Tokenizer.from_pretrained(MODEL)
model = T5ForConditionalGeneration.from_pretrained(MODEL).to(device)
model.eval()

sample = docs[0]["body"] + "</s>"
enc = tok(sample, return_tensors="pt", truncation=True, max_length=1024)
enc = {k: v.to(device) for k, v in enc.items()}
with torch.no_grad():
    out = model.generate(**enc, max_length=256, early_stopping=True)
print(tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True))

In [ ]:
preds = []
for i, doc in enumerate(docs):
    text = doc["body"] + "</s>"
    enc = tok(text, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_length=256, early_stopping=True)
    pred = tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    preds.append({"id": doc["id"], "abstract": doc["abstract"], "pred": pred})
    if (i + 1) % 10 == 0:
        print("done", i + 1)

out_path = Path("/kaggle/working/preds.json")
out_path.write_text(json.dumps(preds, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", out_path, "n=", len(preds))